<a href="https://colab.research.google.com/github/galipaydin/pes3850/blob/main/notebooks/lab05_data_leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 3850 · Lab 5 — How to build a model that looks brilliant and is worthless
**Artificial Intelligence Applications in Sport · Week 5**

⏱️ About 70 minutes · 🧪 This notebook creates its own data, so it will never fail to download anything

### What you will do
You will build an injury-prediction model **four times**. Each time you will remove one mistake, and each time the
performance will get worse. By the end you will have the number that is actually true — and it will be much smaller
than the one you started with. Then, in Stage 5, you will build a model that genuinely works.

| Stage | What we do | AUC you should get |
|---|---|---|
| 1 | The naive way everyone does it first | **≈ 0.93** |
| 2 | Find out why | — |
| 3 | Remove the leaking column | ≈ 0.80 |
| 4 | Split by athlete instead of at random | **≈ 0.69** |
| 5 | Build a model that really works: test on every athlete, use columns with a reason | **≈ 0.71**, and this one you can trust |

> **AUC** (area under the ROC curve) runs from 0.5 (a coin flip) to 1.0 (perfect). Published injury-prediction
> models that are evaluated properly typically land around **0.73–0.77**. Keep that in mind while Stage 1 shows you 0.93.

⚠️ Run the cells in order, top to bottom. Do not skip Stage 2.


---
## Block 1 — Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

RNG = np.random.default_rng(42)
print('Setup complete.')


---
## Block 2 — The data

We simulate one season for a squad of 40 athletes, 30 weeks each: **1,200 athlete-weeks**.

Because we built it, we know the truth: injury risk really does depend on the workload spike (`acwr`), soreness,
sleep and each athlete's own underlying fragility. That fragility is **not** in the data — just as in real life.

One extra column, `days_missed_next_week`, is included. It comes from the club's attendance system.
Remember that it is there.


In [ ]:
N_ATH, N_WK = 40, 30

athletes = pd.DataFrame({
    'athlete_id':     [f'A{i:02d}' for i in range(N_ATH)],
    'age':            RNG.integers(18, 34, N_ATH),
    'height_cm':      np.round(RNG.normal(180, 8, N_ATH), 1),
    'resting_hr':     RNG.integers(42, 62, N_ATH),
    'prior_injuries': RNG.poisson(1.2, N_ATH),
})
athletes['frailty'] = RNG.normal(0, 1.0, N_ATH)   # real, but never measured

rows = []
for _, a in athletes.iterrows():
    load    = RNG.normal(2600, 500, N_WK).clip(800, None)
    chronic = pd.Series(load).rolling(4, min_periods=1).mean().values
    acwr    = load / chronic
    sleep   = RNG.normal(7.4, 0.9, N_WK).clip(4, 10)
    welln   = RNG.integers(2, 6, N_WK)
    sore    = RNG.integers(1, 6, N_WK)
    lin = (-3.9 + 0.85*a['frailty'] + 1.5*(acwr - 1.0) + 0.28*sore
           - 0.22*(sleep - 7.4) + 0.18*a['prior_injuries'])
    injury = RNG.binomial(1, 1/(1 + np.exp(-lin)))
    for w in range(N_WK):
        rows.append(dict(athlete_id=a['athlete_id'], week=w+1, age=a['age'],
                         height_cm=a['height_cm'], resting_hr=a['resting_hr'],
                         prior_injuries=a['prior_injuries'],
                         weekly_load=round(load[w]), acwr=round(acwr[w], 3),
                         sleep_h=round(sleep[w], 2), wellness=welln[w],
                         soreness=sore[w], injury=int(injury[w])))

df = pd.DataFrame(rows)

# the club's attendance export: days missed in the week after this one
# not every injury costs days (some are minor); and some absences have nothing to do with injury
_from_injury = np.where(df.injury == 1,
                        RNG.integers(1, 15, len(df)) * (RNG.random(len(df)) < 0.70), 0)
_other       = (RNG.random(len(df)) < 0.05) * RNG.integers(1, 6, len(df))  # illness, personal
df['days_missed_next_week'] = np.maximum(_from_injury, _other)

print(f'{len(df)} athlete-weeks, {df.athlete_id.nunique()} athletes')
print(f'Injury rate: {df.injury.mean():.1%}')
df.head()


---
## Block 3 — The honest baseline

**Before any model, ask what a rule with no intelligence in it would score.**
If you cannot beat this, you have nothing.


In [ ]:
naive_accuracy = 1 - df.injury.mean()
print(f'Rule: "nobody will get injured this week"')
print(f'  Accuracy : {naive_accuracy:.1%}   <-- looks excellent')
print(f'  AUC      : 0.500        <-- and is completely useless')
print(f'  Injuries caught: 0 of {int(df.injury.sum())}')


> ✋ **Stop and note this.** Any model in this notebook that reports ~93% accuracy has learned nothing.
> This is why accuracy is the wrong metric for a rare outcome, and why we use AUC instead.


---
# STAGE 1 — The naive model

Throw every column at a random forest. Split the rows 75/25 at random. Report the AUC.
This is, honestly, what most first attempts look like.


In [ ]:
FEATURES_ALL = ['age','height_cm','resting_hr','prior_injuries','weekly_load',
                'acwr','sleep_h','wellness','soreness','days_missed_next_week']

X, y = df[FEATURES_ALL], df['injury']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)

m1 = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
m1.fit(Xtr, ytr)
p1 = m1.predict_proba(Xte)[:, 1]
auc1 = roc_auc_score(yte, p1)

print(f'STAGE 1 AUC = {auc1:.3f}')
print('\nThis would get published. It would also get sold to your club.')


🎉 **Congratulations, you have built a state-of-the-art injury prediction model.**

Now spend two minutes deciding whether you believe it, before you run the next cell.


---
# STAGE 2 — Find out why

Ask the model which columns it is actually using.


In [ ]:
imp = (pd.Series(m1.feature_importances_, index=FEATURES_ALL)
         .sort_values(ascending=True))

plt.figure(figsize=(7, 4.5))
colors = ['crimson' if i == 'days_missed_next_week' else 'steelblue' for i in imp.index]
plt.barh(imp.index, imp.values, color=colors)
plt.xlabel('Importance to the model')
plt.title('What is the model looking at?')
plt.tight_layout(); plt.show()

print(imp.sort_values(ascending=False).round(3))


### The problem

`days_missed_next_week` is the column doing the work. Look at what it contains:

An athlete misses days next week **because** they got injured this week. The column is not a *predictor* of injury —
it is a **consequence** of it. The model has not learned to see an injury coming. It has learned to read the
attendance record afterwards.

This is **data leakage**: information that would not be available at the moment of the real prediction has ended up
in the training data.

It is almost never as obvious as this in practice. In a real club export it arrives as `availability_status`,
`sessions_completed`, `medical_flag`, `RTP_stage` or `minutes_next_match` — all of which are quietly written
*after* the thing you are trying to predict.


In [ ]:
print(df.groupby('injury')['days_missed_next_week'].describe()[['count','mean','max']].round(2))


---
# STAGE 3 — Remove the leaking column

Same model, same random split, one column dropped.


In [ ]:
FEATURES_CLEAN = [f for f in FEATURES_ALL if f != 'days_missed_next_week']

X, y = df[FEATURES_CLEAN], df['injury']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)

m3 = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
m3.fit(Xtr, ytr)
auc3 = roc_auc_score(yte, m3.predict_proba(Xte)[:, 1])

print(f'STAGE 1 AUC (leaky) = {auc1:.3f}')
print(f'STAGE 3 AUC (clean) = {auc3:.3f}')
print(f'Cost of one leaking column: {auc1 - auc3:.3f} AUC')


### But this is still not honest

We split **1,200 rows** at random. Those rows are not 1,200 independent observations — they are 40 athletes
observed 30 times each.

A random split puts *the same athlete* in both the training set and the test set. The model does not have to learn
anything about injury risk in general; it can learn that **athlete A17 is fragile**, and then recognise A17 in the
test set from their height, resting heart rate and age.

That is not prediction. That is recognition. And it tells you nothing about the new signing arriving in January.


---
# STAGE 4 — Split by athlete

`GroupShuffleSplit` guarantees that every athlete is entirely in the training set or entirely in the test set — never both.


In [ ]:
groups = df['athlete_id']
gss = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=0)
tr_idx, te_idx = next(gss.split(X, y, groups))

print(f'Athletes in training: {groups.iloc[tr_idx].nunique()}')
print(f'Athletes in test    : {groups.iloc[te_idx].nunique()}')
print(f'Overlap             : {len(set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx]))} athletes\n')

m4 = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
m4.fit(X.iloc[tr_idx], y.iloc[tr_idx])
p4 = m4.predict_proba(X.iloc[te_idx])[:, 1]
auc4 = roc_auc_score(y.iloc[te_idx], p4)

print(f'STAGE 4 AUC (clean + grouped by athlete) = {auc4:.3f}')


In [ ]:
stages = ['1. Leaky\n(random split)', '3. Clean\n(random split)',
          '4. Clean\n(grouped split)', 'Coin flip']
vals   = [auc1, auc3, auc4, 0.5]

plt.figure(figsize=(7.5, 4.5))
bars = plt.bar(stages, vals, color=['crimson', 'orange', 'seagreen', 'lightgrey'])
plt.axhline(0.5, ls='--', c='grey')
plt.axhspan(0.73, 0.77, alpha=.15, color='navy',
            label='typical honest range in the published literature')
plt.ylim(0.4, 1.0); plt.ylabel('AUC')
plt.title('The same data, the same model, four different truths')
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width()/2, v + .012, f'{v:.3f}', ha='center', fontweight='bold')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()


---
## Block 6 — Break it 💥

One more, for the road. Change `TEST_SIZE` below to `0.02` and re-run: the test set now contains about
24 athlete-weeks and roughly 2 injuries. Watch the AUC swing wildly between random seeds.

**A number computed on a tiny test set is noise with a decimal point.**


In [ ]:
TEST_SIZE = 0.02          # ✏️ EDIT ME — try 0.02, then 0.10, then 0.25

aucs = []
for seed in range(8):
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    tr, te = next(gss.split(X, y, groups))
    if y.iloc[te].nunique() < 2:
        continue
    mm = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=0, n_jobs=-1)
    mm.fit(X.iloc[tr], y.iloc[tr])
    aucs.append(roc_auc_score(y.iloc[te], mm.predict_proba(X.iloc[te])[:, 1]))

if len(aucs) < 2:
    print('The test set was so small that most splits contained no injuries at all.')
    print('That is itself the lesson: raise TEST_SIZE and try again.')
else:
    print(f'test_size = {TEST_SIZE}')
    print('AUC across 8 random seeds:', [round(a, 3) for a in aucs])
    print(f'range = {max(aucs) - min(aucs):.3f}   <-- the same model, the same data')


---
# STAGE 5 — Build a model that really works

Stages 1–4 were about **removing** mistakes. This stage is about **adding sense**. Three steps:

1. **Judge on every athlete, not one split.** Stage 4 tested on 10 athletes. Block 6 just showed you how much a
   number moves when the test set changes. From now on every athlete is predicted once, by a model that never saw them.
2. **Use only columns with a reason.** Age, height and resting heart rate mostly tell the model *who* an athlete is.
   Workload spike, soreness, sleep and injury history have a reason to matter.
3. **Use what the club already knows on Monday morning.** How many times has this athlete been injured *so far this
   season*? That is not leakage: it is counted only from earlier weeks.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.calibration import calibration_curve

d  = df.sort_values(['athlete_id', 'week']).reset_index(drop=True)
cv = GroupKFold(n_splits=5)          # 5 groups of 8 athletes; each group is the test set once

def honest_scores(features, model):
    # every athlete-week gets a risk score from a model trained WITHOUT that athlete
    return cross_val_predict(model, d[features], d['injury'], groups=d['athlete_id'],
                             cv=cv, method='predict_proba')[:, 1]

forest   = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
p_forest = honest_scores(FEATURES_CLEAN, forest)
auc_forest = roc_auc_score(d['injury'], p_forest)

print(f'Stage 4 forest, tested on one split of 10 athletes : AUC {auc4:.3f}')
print(f'Same forest, tested on all 40 athletes (5 folds)   : AUC {auc_forest:.3f}')

### Step 1 — the Stage 4 number was partly luck

Tested on all 40 athletes, the forest falls close to a coin flip. The single split happened to be kind to it.
A random forest with ten columns and only 30 training athletes has plenty of room to chase noise.

### Step 2 — fewer columns, each with a reason, and a simpler model

Logistic regression draws one smooth line through risk instead of hundreds of yes/no splits. With little data,
simple usually wins.

In [ ]:
REASONED = ['acwr', 'soreness', 'sleep_h', 'prior_injuries']

simple     = make_pipeline(StandardScaler(), LogisticRegression())
p_simple   = honest_scores(REASONED, simple)
auc_simple = roc_auc_score(d['injury'], p_simple)

print(f'Forest, all 9 clean columns      : AUC {auc_forest:.3f}')
print(f'Logistic regression, 4 columns   : AUC {auc_simple:.3f}')

### Step 3 — what the club already knows about each athlete

Some athletes are simply more fragile, and that fragility is not in any column. But it leaves a trace: they get
injured more often. Count each athlete's injuries **in the weeks before this one** and the model can see it.

`shift()` is the important word below: it moves the count down one week, so week 12 only knows about weeks 1–11.
Without it, this column would leak exactly like `days_missed_next_week`.

In [ ]:
d['injuries_so_far'] = (d.groupby('athlete_id')['injury']
                          .transform(lambda s: s.shift(fill_value=0).cumsum()))
d['weeks_so_far']    = d['week'] - 1

# check it: one athlete, the weeks around an injury. The count goes up the week AFTER, never the same week.
first_injured = d.loc[d.injury == 1, 'athlete_id'].iloc[0]
wk = d.loc[(d.athlete_id == first_injured) & (d.injury == 1), 'week'].iloc[0]
d.loc[(d.athlete_id == first_injured) & d.week.between(wk - 1, wk + 2),
      ['athlete_id', 'week', 'injury', 'injuries_so_far']]

In [ ]:
FINAL = REASONED + ['injuries_so_far', 'weeks_so_far']

p_final   = honest_scores(FINAL, make_pipeline(StandardScaler(), LogisticRegression()))
auc_final = roc_auc_score(d['injury'], p_final)
base      = d['injury'].mean()

table = pd.DataFrame({
    'model': ['Stage 4 forest, 9 columns', 'Logistic, 4 columns with a reason', '+ injury history so far'],
    'ROC-AUC': [auc_forest, auc_simple, auc_final],
    'PR-AUC':  [average_precision_score(d.injury, p) for p in (p_forest, p_simple, p_final)],
}).round(3)
print(f'Every number below: 40 athletes, 5 athlete-grouped folds. PR-AUC floor (guessing) = {base:.3f}\n')
print(table.to_string(index=False))

plt.figure(figsize=(7.5, 4.2))
bars = plt.bar(['Stage 4\nforest', 'Logistic\n4 columns', '+ history\nso far', 'Coin flip'],
               [auc_forest, auc_simple, auc_final, 0.5], color=['orange', 'steelblue', 'seagreen', 'lightgrey'])
plt.axhspan(0.73, 0.77, alpha=.15, color='navy', label='typical honest range in the published literature')
for b, v in zip(bars, [auc_forest, auc_simple, auc_final, 0.5]):
    plt.text(b.get_x() + b.get_width()/2, v + .01, f'{v:.3f}', ha='center', fontweight='bold')
plt.ylim(0.4, 0.9); plt.ylabel('ROC-AUC (all 40 athletes)'); plt.legend(loc='upper left', fontsize=8)
plt.title('Adding sense, not columns'); plt.tight_layout(); plt.show()

### Does it work? Three checks

A good ROC-AUC is not enough. Look at the curves, and check that the percentages mean what they say.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.3))

for p, name, c in [(p_final, f'final model ({auc_final:.2f})', 'seagreen'),
                   (p_forest, f'Stage 4 forest ({auc_forest:.2f})', 'orange')]:
    fpr, tpr, _ = roc_curve(d.injury, p); ax[0].plot(fpr, tpr, c=c, lw=2, label=name)
ax[0].plot([0, 1], [0, 1], '--', c='grey', label='coin flip (0.50)')
ax[0].set(xlabel='false alarms (share of healthy weeks flagged)', ylabel='injuries caught', title='ROC curve')
ax[0].legend(fontsize=8)

from sklearn.metrics import precision_recall_curve
for p, c in [(p_final, 'seagreen'), (p_forest, 'orange')]:
    pr, rc, _ = precision_recall_curve(d.injury, p); ax[1].plot(rc, pr, c=c, lw=2)
ax[1].axhline(base, ls='--', c='grey'); ax[1].text(0.55, base + .03, f'guessing = {base:.2f}', color='grey')
ax[1].set(xlabel='injuries caught', ylabel='share of flags that are real', title='Precision-recall', ylim=(0, .6))

obs, pred = calibration_curve(d.injury, p_final, n_bins=5, strategy='quantile')
ax[2].plot([0, .3], [0, .3], '--', c='grey', label='perfect')
ax[2].plot(pred, obs, 'o-', c='seagreen', lw=2, ms=8, label='final model')
ax[2].set(xlabel='risk the model predicted', ylabel='share really injured', title='Calibration')
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

### What the medical room would actually see

A model does not flag anyone until somebody picks a rule. A realistic one: **each week, look closely at the 4
athletes (10% of the squad) with the highest risk.** How does that compare with picking 4 at random?

In [ ]:
N_FLAG = 4      # ✏️ EDIT ME — try 2, 4, 8: more flags catch more injuries and waste more time

d['risk'] = p_final
d['flag'] = d.groupby('week')['risk'].rank(ascending=False, method='first') <= N_FLAG

tn, fp, fn, tp = confusion_matrix(d.injury, d.flag).ravel()
caught_random  = d.injury.sum() * N_FLAG / N_ATH

print(f'Flagging the top {N_FLAG} of {N_ATH} athletes every week, for {N_WK} weeks:')
print(f'  injuries caught      : {tp} of {tp + fn}  (sensitivity {tp / (tp + fn):.2f})')
print(f'  random {N_FLAG} per week   : about {caught_random:.0f} of {tp + fn}')
print(f'  flags that were real : {tp} of {tp + fp}  (precision {tp / (tp + fp):.2f}; random = {base:.2f})')
print(f'\nIn plain words: about 1 flag in {round((tp + fp) / max(tp, 1))} is a real injury. '
      f'That is a reason to look more closely, not a diagnosis.')

In [ ]:
# what pushes risk up? Final model trained on everyone; coefficients on a common scale
final_model = make_pipeline(StandardScaler(), LogisticRegression()).fit(d[FINAL], d['injury'])
coef = pd.Series(final_model[-1].coef_[0], index=FINAL).sort_values()

plt.figure(figsize=(7, 3.8))
plt.barh(coef.index, coef.values, color=['steelblue' if v < 0 else 'crimson' for v in coef.values])
plt.axvline(0, c='grey'); plt.xlabel('← lowers risk        raises risk →')
plt.title('What the final model learned'); plt.tight_layout(); plt.show()

> **Compare with how the data was made (Block 2):** risk was built from the workload spike, soreness, sleep, prior
> injuries and a hidden fragility. The model found the same things, in the same directions: sleep pushes risk *down*;
> the workload spike, soreness and injuries push it *up*. `injuries_so_far` stands in for the fragility nobody measured,
> and `weeks_so_far` points down because two injuries in 25 weeks is a sturdier athlete than two in 5.
>
> **This is what a model that works looks like in sport: ROC-AUC near 0.7, a short list each week that catches several
> times more injuries than chance, and percentages you can trust.** Not 0.93.

---
## Block 8 — AI Use Log

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
## What to take out of this lab

1. **Always compute the trivial baseline first.** 93% accuracy meant nothing here.
2. **Ask what each column means and when it was written.** Anything recorded after the outcome is leakage.
3. **Repeated measurements on the same people are not independent rows.** Split by athlete, not by row.
4. **A high number is a reason for suspicion, not celebration.** If an injury model beats AUC 0.80, find out why before you believe it.
5. **A model that works is built from sense, not columns.** Few features with a reason, a simple model, history counted only from the past.

You will apply all four of these to your own term project, where 25 of the 100 marks are for exactly this.
